<h1>Fine tuning Qwen 4B Parameters</h1>

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_ALLOW_CODE_EVAL"] = "1"

<h3>Loading the dataset</h3>

In [2]:
from datasets import load_dataset
dataset_path = "dataset/dataset_curation_agent/valid_codes.jsonl"
dataset = load_dataset("json", data_files=dataset_path, split="train")

dataset = dataset.shuffle(seed=42)

total_len = len(dataset)
train_end = int(0.7 * total_len)
eval_end = int(0.9 * total_len)

# Slice the dataset
train_dataset = dataset.select(range(0, train_end))
eval_dataset  = dataset.select(range(train_end, eval_end))
test_dataset  = dataset.select(range(643, 653))

# Print lengths to verify
print("Total:", total_len)
print("Train:", len(train_dataset))
print("Eval:", len(eval_dataset))
print("Test:", len(test_dataset))
print("\n")

print("-------------------------------------\nTrain:",len(train_dataset),"example: ",train_dataset[0],"\n-------------------------------------\n")
print("-------------------------------------\nEval:", len(eval_dataset),"\n example: ",eval_dataset[0],"\n-------------------------------------\n")
print("-------------------------------------\nTest:", len(test_dataset),"\n example: ",test_dataset[0],"\n-------------------------------------\n")

EVAL_REFERENCES = [ex["correct_code"] for ex in eval_dataset]
TEST_REFERENCES = [ex["correct_code"] for ex in test_dataset]
print("eval_references:", EVAL_REFERENCES[0],"\n")
print("test_references:", TEST_REFERENCES[0],"\n")

Total: 654
Train: 457
Eval: 131
Test: 10


-------------------------------------
Train: 457 example:  {'task': 'Fix the issue in the following Python code.', 'buggy_code': 'def _u_in(self, u):\n    return u >= 0.0 or u <= 1.0', 'correct_code': 'def _u_in(self, u):\n    return u >= 0.0 and u <= 1.0', 'unit_test': 'def check(candidate):\n    # Test cases for numbers within the range [0.0, 1.0]\n    assert candidate(0.0) == True\n    assert candidate(1.0) == True\n    assert candidate(0.5) == True\n    \n    # Test cases for numbers outside the range\n    assert candidate(-0.1) == False\n    assert candidate(1.1) == False\n    \n    # Edge case: exactly at the boundaries\n    assert candidate(0.0) == True  # Lower boundary\n    assert candidate(1.0) == True  # Upper boundary\n\n    # Test cases for numbers equal to the boundaries\n    assert candidate(-0.0) == True  # -0.0 is equivalent to 0.0 in Python\n    \n    # Additional test case with a number very close to the boundaries\n    asse

In [3]:
# Just for your train split
def formatting_prompts_func(examples):
    output_text = []
    for i in range(len(examples["task"])):
        task = examples["task"][i]
        buggy_code = examples["buggy_code"][i]
        correct_code = examples["correct_code"][i]

        if buggy_code.strip():
            text = f"""### Instruction:
            {task}

            ### Buggy Code:
            {buggy_code}

            ### Fixed Code:
            {correct_code}
            """
            output_text.append(text)
    return output_text

In [4]:
import torch
cuda_available = torch.cuda.is_available()

if cuda_available:
    device_id = 0  # You can change to 1,2,3 if you want other GPUs
    torch.cuda.set_device(device_id)
    # device = torch.device(f"cuda:{device_id}")
    device = torch.device(f"cuda:{device_id}")
    print(f"🖥️ Using GPU {device_id}: {torch.cuda.get_device_name(device_id)}")
else:
    device = torch.device("cpu")
    print("⚙️ No GPU available, using CPU.")

print(f"Device selected: {device}")

🖥️ Using GPU 0: NVIDIA GeForce RTX 4070 SUPER
Device selected: cuda:0


<h3>Importing the LLM(Qwen3)</h3>

In [5]:
from unsloth import FastLanguageModel
base_model_name="unsloth/Qwen3-4B-unsloth-bnb-4bit"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = base_model_name,
    max_seq_length = 512,   # Context length - can be longer, but uses more memory
    load_in_4bit = True,     # 4bit uses much less memory
    load_in_8bit = False,    # A bit more accurate, uses 2x memory
    full_finetuning = False, # We have full finetuning now!
    # token = "hf_...",      # use one if using gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.5.7: Fast Qwen3 patching. Transformers: 4.51.3.
   \\   /|    NVIDIA GeForce RTX 4070 SUPER. Num GPUs = 2. Max memory: 11.994 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.3.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 64,           # Choose any number > 0! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 64,  # Best to choose alpha = rank or rank*2
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = True,   # We support rank stabilized LoRA
    loftq_config = None,  # And LoftQ
)

Unsloth 2025.5.7 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


<h3>Test Evaluation Before Training</h3>

In [7]:
from tqdm import tqdm
import evaluate
import torch  # Make sure torch is imported

def evaluate_pass_at_k(model, tokenizer, prompts, references, k_values=[1, 5, 10], num_completions=10, max_new_tokens=256):
    code_eval = evaluate.load("code_eval")

    all_predictions = []

    model.eval()
    for prompt in tqdm(prompts, desc="Generating Completions"):
        input_ids = tokenizer(prompt, return_tensors="pt").input_ids.cuda()
        outputs = model.generate(
            input_ids=input_ids,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.7,
            num_return_sequences=num_completions,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id
        )
        torch.cuda.empty_cache()
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        cleaned = []
        for d in decoded:
            parts = d.split("### Fixed Code:")
            cleaned.append(parts[-1].strip() if len(parts) > 1 else d.strip())

        all_predictions.append(cleaned)

    print("\n All completions generated. Computing pass@k...\n")
    result, _ = code_eval.compute(
        references=references,
        predictions=all_predictions,
        k=k_values,
    )

    print("🎯 Final pass@k scores:")
    for k in k_values:
        score = result.get(f'pass@{k}', 'N/A')
        if isinstance(score, (float, int)):
            print(f"pass@{k}: {score:.4f}")
        else:
            print(f"pass@{k}: {score}")

    return result

# Generating prompts
prompts = []
for ex in test_dataset:
    prompt = f"""Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{ex["task"]}

### Input:
{ex["buggy_code"]}

### Response:"""
    prompts.append(prompt)

# Evaluate and print returned pass@k scores
pass_at_k_scores = evaluate_pass_at_k(model, tokenizer, prompts, TEST_REFERENCES)
print("Final pass@k scores:")
print(pass_at_k_scores)

Generating Completions:   0%|          | 0/10 [00:00<?, ?it/s]The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
/home/carlos/projects/Code-Fixer-LLM-Agent/fine-tuning-LLM/.venv/lib/python3.12/site-packages/unsloth/kernels/utils.py:438: UserWarning: An output with one or more elements was resized since it had shape [1, 10, 2560], which does not match the required output shape [10, 1, 2560]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /pytorch/aten/src/ATen/native/Resize.cpp:30.)
  out = torch_matmul(X, W.t(), out = out)
/home/carlos/projects/Code-Fixer-LLM-Agent/fine-tuning-LLM/.venv/lib/python3.12/site-package


 All completions generated. Computing pass@k...



🎯 Final pass@k scores:
pass@1: 0.0000
pass@5: 0.0000
pass@10: 0.0000
Final pass@k scores:
{'pass@1': np.float64(0.0), 'pass@5': np.float64(0.0), 'pass@10': np.float64(0.0)}


In [8]:
import evaluate
from codebleu import compute_codebleu

# Metrics
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
acc = evaluate.load("accuracy")
code_eval = evaluate.load("code_eval")

def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]
    return logits.argmax(dim=-1)

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    labels = labels[:, 1:]
    preds = preds[:, :-1]

    # Mask handling
    mask = labels == -100
    labels[mask] = tokenizer.pad_token_id
    preds[mask] = tokenizer.pad_token_id

    # Decode
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    decoded_completions = []
    for pred in decoded_preds:
        parts = pred.split("### Fixed Code:")
        completion = parts[-1].strip() if len(parts) > 1 else pred.strip()
        decoded_completions.append(completion)

    # Standard metrics
    bleu_score = bleu.compute(predictions=decoded_completions, references=EVAL_REFERENCES)
    rouge_score = rouge.compute(predictions=decoded_completions, references=EVAL_REFERENCES)
    accuracy = acc.compute(predictions=preds[~mask], references=labels[~mask])
    refs = [[ref] for ref in EVAL_REFERENCES]
    codebleu_scores = compute_codebleu(decoded_completions, refs, lang="python")

    # # Simulate pass@1: just one candidate per sample
    # predictions_for_code_eval = [[c] for c in decoded_completions]
    # pass_at_k, _ = code_eval.compute(
    #     references=EVAL_REFERENCES,
    #     predictions=predictions_for_code_eval,
    #     k=[1,5,10],
    # )

    return {
        "codebleu": codebleu_scores["codebleu"],
        **bleu_score,
        **rouge_score,
        **accuracy
    }

<h3>Setting up Neptune.ai</h3>

In [9]:
import neptune
from transformers.integrations import NeptuneCallback
run = neptune.init_run(
    project="casvi/CodeMedic",
    api_token="eyJhcGlfYWRkcmVzcyI6Imh0dHBzOi8vYXBwLm5lcHR1bmUuYWkiLCJhcGlfdXJsIjoiaHR0cHM6Ly9hcHAubmVwdHVuZS5haSIsImFwaV9rZXkiOiIzMTMzYjhhOC1jYzA1LTQ0YjAtOTJjNi1iY2EzM2VhMDY0OTcifQ=="
)

[neptune] [warning] NeptuneWarning: By default, these monitoring options are disabled in interactive sessions: 'capture_stdout', 'capture_stderr', 'capture_traceback', 'capture_hardware_metrics'. You can set them to 'True' when initializing the run and the monitoring will continue until you call run.stop() or the kernel stops. NOTE: To track the source files, pass their paths to the 'source_code' argument. For help, see: https://docs-legacy.neptune.ai/logging/source_code/


[neptune] [info   ] Neptune initialized. Open in the app: https://app.neptune.ai/casvi/CodeMedic/e/COD-136


<h3>Fine tuning the LLM</h3>

In [10]:
from trl import SFTTrainer, SFTConfig
import time

output_dir='./results'
logging_dir='./logs'

learning_rate =1.9680e-4
num_epochs =6
batch_size=2
steps_per_epoch = len(train_dataset) // batch_size
max_steps = num_epochs * steps_per_epoch

start=time.time()
# SFT Config
config = SFTConfig(
    dataset_num_proc = 1,
    output_dir=output_dir,
    logging_dir=logging_dir,
    eval_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    dataset_text_field="text",
    learning_rate=learning_rate,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=batch_size,
    num_train_epochs=num_epochs,
    report_to="none",
    save_steps=200,
    logging_steps=200,
    max_steps=max_steps,
    eval_accumulation_steps=100
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    formatting_func=formatting_prompts_func,
    args=config,
    warmup_steps = 5,
    weight_decay = 0.01,
    compute_metrics = compute_metrics,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
)
metrics_before_training = trainer.evaluate()
print("Metrics before training:", metrics_before_training)
run["eval/before_training/"] = metrics_before_training

trainer.train()

for record in trainer.state.log_history:
    if "loss" in record:
        step = record.get("step", None)
        loss = record["loss"]
        run["train/loss"].append({"step": step, "value": loss})

metrics_after_training = trainer.evaluate()
print("Metrics after training:", metrics_after_training)
run["eval/after_training/"] = metrics_after_training

run.stop()
end = time.time()
length = end - start

hours = int(length // 3600)
minutes = int((length % 3600) // 60)
seconds = int(length % 60)

print(f"It took {hours} hours, {minutes} minutes, and {seconds} seconds to train the model!")

Unsloth: Not an error, but Qwen3ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Metrics before training: {'eval_loss': 1.5198644399642944, 'eval_model_preparation_time': 0.0056, 'eval_codebleu': 0.5506996833434399, 'eval_bleu': 0.47804520618492374, 'eval_precisions': [0.5351969504447268, 0.487862451752847, 0.45797296431545903, 0.4367420465146203], 'eval_brevity_penalty': 1.0, 'eval_length_ratio': 1.5911847957945815, 'eval_translation_length': 31480, 'eval_reference_length': 19784, 'eval_rouge1': 0.5955486131532799, 'eval_rouge2': 0.5499658277959119, 'eval_rougeL': 0.5672345510396635, 'eval_rougeLsum': 0.593783329249328, 'eval_accuracy': 0.7436515997968512, 'eval_runtime': 17.705, 'eval_samples_per_second': 7.399, 'eval_steps_per_second': 0.96}


[neptune] [warning] NeptuneUnsupportedType: You're attempting to log a type that is not directly supported by Neptune (<class 'list'>).
        Convert the value to a supported type, such as a string or float, or use stringify_unsupported(obj)
        for dictionaries or collections that contain unsupported values.
        For more, see https://docs-legacy.neptune.ai/help/value_of_unsupported_type
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 457 | Num Epochs = 24 | Total steps = 1,368
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 132,120,576/4,000,000,000 (3.30% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss,Model Preparation Time,Codebleu,Bleu,Precisions,Brevity Penalty,Length Ratio,Translation Length,Reference Length,Rouge1,Rouge2,Rougel,Rougelsum,Accuracy
200,0.487100,1.247142,0.005600,0.647794,0.642577,"[0.972757292239956, 0.9291218326969802, 0.9030404932044276, 0.885314289754649]",0.696955,0.734735,14536,19784,0.928668,0.913790,0.924965,0.928503,0.797550
400,0.094700,1.339675,0.005600,0.645731,0.641460,"[0.9664204623170565, 0.920433372438065, 0.8930362116991644, 0.8748330873568065]",0.702556,0.739082,14622,19784,0.925064,0.910327,0.921645,0.924931,0.793233
600,0.054700,1.415402,0.005600,0.646523,0.644686,"[0.9571205644887713, 0.914841182913472, 0.8894107895282172, 0.8721594869650077]",0.710143,0.744996,14739,19784,0.923159,0.909416,0.919717,0.922823,0.795423
800,0.036900,1.579308,0.005600,0.650017,0.641313,"[0.9727103576597065, 0.9296244784422809, 0.9032914590497578, 0.8859611843037257]",0.695323,0.733471,14511,19784,0.930434,0.917642,0.926982,0.929945,0.796280
1000,0.030900,1.662006,0.005600,0.649315,0.641683,"[0.9727178780571822, 0.9295745272525028, 0.9036694029327159, 0.8863475428409574]",0.695584,0.733674,14515,19784,0.930048,0.916321,0.926465,0.929715,0.796058
1200,0.029800,1.695700,0.005600,0.649252,0.641686,"[0.9728612756578041, 0.9293111837075138, 0.9032687991021324, 0.8858761061946903]",0.695780,0.733825,14518,19784,0.930097,0.916317,0.926463,0.929716,0.796819


Metrics after training: {'eval_loss': 1.2471420764923096, 'eval_model_preparation_time': 0.0056, 'eval_codebleu': 0.6477935415406673, 'eval_bleu': 0.6425768849091438, 'eval_precisions': [0.972757292239956, 0.9291218326969802, 0.9030404932044276, 0.885314289754649], 'eval_brevity_penalty': 0.6969548328556001, 'eval_length_ratio': 0.734735139506672, 'eval_translation_length': 14536, 'eval_reference_length': 19784, 'eval_rouge1': 0.9286675760741647, 'eval_rouge2': 0.9137895103440667, 'eval_rougeL': 0.9249654497159108, 'eval_rougeLsum': 0.9285034628552727, 'eval_accuracy': 0.7975495175215845, 'eval_runtime': 12.5271, 'eval_samples_per_second': 10.457, 'eval_steps_per_second': 1.357}
[neptune] [info   ] Shutting down background jobs, please wait a moment...
[neptune] [info   ] Done!
[neptune] [info   ] Waiting for the remaining 16 operations to synchronize with Neptune. Do not kill this process.
[neptune] [info   ] All 16 operations synced, thanks for waiting!
[neptune] [info   ] Explore th

<h3>Saving the model</h3>

In [11]:
import os
model.save_pretrained(os.path.join(output_dir, "final_checkpoint"))
tokenizer.save_pretrained(os.path.join(output_dir, "final_checkpoint"))

('./results/final_checkpoint/tokenizer_config.json',
 './results/final_checkpoint/special_tokens_map.json',
 './results/final_checkpoint/vocab.json',
 './results/final_checkpoint/merges.txt',
 './results/final_checkpoint/added_tokens.json',
 './results/final_checkpoint/tokenizer.json')

<h3>Evaluate fine tuned model</h3>

In [3]:
from unsloth import FastLanguageModel
from transformers import AutoTokenizer
from peft import PeftModel
import os
from tqdm import tqdm
import evaluate
import torch

os.environ["HF_ALLOW_CODE_EVAL"] = "1"
# === Paths ===

base_model_name="unsloth/Qwen3-4B-unsloth-bnb-4bit"
output_dir='./results'

base_model, _ = FastLanguageModel.from_pretrained(
    model_name = base_model_name,
    max_seq_length = 512,
    load_in_4bit = True,
)

adapter_path = os.path.join(output_dir, "final_checkpoint")
tokenizer = AutoTokenizer.from_pretrained(adapter_path, trust_remote_code=True)
model = PeftModel.from_pretrained(base_model, adapter_path)



def evaluate_pass_at_k(model, tokenizer, prompts, references, k_values=[1, 5, 10], num_completions=10, max_new_tokens=1024):
    code_eval = evaluate.load("code_eval")

    all_predictions = []

    model.eval()
    for prompt in tqdm(prompts, desc="Generating Completions"):
        input_ids = tokenizer(prompt, return_tensors="pt").input_ids.cuda()
        outputs = model.generate(
            input_ids=input_ids,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.7,
            num_return_sequences=num_completions,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id
        )
        torch.cuda.empty_cache()
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        cleaned = []
        for d in decoded:
            parts = d.split("### Fixed Code:")
            cleaned.append(parts[-1].strip() if len(parts) > 1 else d.strip())

        all_predictions.append(cleaned)

    print("\n All completions generated. Computing pass@k...\n")
    result, _ = code_eval.compute(
        references=references,
        predictions=all_predictions,
        k=k_values,
    )

    print("🎯 Final pass@k scores:")
    for k in k_values:
        score = result.get(f'pass@{k}', 'N/A')
        if isinstance(score, (float, int)):
            print(f"pass@{k}: {score:.4f}")
        else:
            print(f"pass@{k}: {score}")

    return result

# Generating prompts
prompts = []
for ex in test_dataset:
    prompt = f"""Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{ex["task"]}

### Input:
{ex["buggy_code"]}

### Response:"""
    prompts.append(prompt)

# Evaluate and print returned pass@k scores
pass_at_k_scores = evaluate_pass_at_k(model, tokenizer, prompts, TEST_REFERENCES)
print("Final pass@k scores:")
print(pass_at_k_scores)


==((====))==  Unsloth 2025.5.7: Fast Qwen3 patching. Transformers: 4.51.3.
   \\   /|    NVIDIA GeForce RTX 4070 SUPER. Num GPUs = 2. Max memory: 11.994 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.3.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Generating Completions:   0%|          | 0/10 [00:00<?, ?it/s]The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
/home/carlos/projects/Code-Fixer-LLM-Agent/fine-tuning-LLM/.venv/lib/python3.12/site-packages/unsloth/kernels/utils.py:438: UserWarning: An output with one or more elements was resized since it had shape [1, 10, 2560], which does not match the required output shape [10, 1, 2560]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /pytorch/aten/src/ATen/native/Resize.cpp:30.)
  out = torch_matmul(X, W.t(), out = out)
/home/carlos/projects/Code-Fixer-LLM-Agent/fine-tuning-LLM/.venv/lib/python3.12/site-package


 All completions generated. Computing pass@k...




huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The

🎯 Final pass@k scores:
pass@1: 0.2700
pass@5: 0.6008
pass@10: 0.7000
Final pass@k scores:
{'pass@1': np.float64(0.27), 'pass@5': np.float64(0.6007936507936508), 'pass@10': np.float64(0.7)}


<h3>Save model in hugging face</h3>

In [2]:
import os
from dotenv import load_dotenv
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from unsloth import FastLanguageModel
output_dir='./results'
model_path = os.path.join(output_dir, "final_checkpoint")
print("model_path: ",model_path)
model = AutoModelForCausalLM.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)



dotenv_path = ".env"
load_dotenv(dotenv_path=dotenv_path)
HUGGINGFACEHUB_API_TOKEN = os.getenv("HUGGINGFACEHUB_API_TOKEN")


base_model_name = "unsloth/Qwen3-4B-unsloth-bnb-4bit"
adapter_path = "./results/final_checkpoint"

base_model, _ = FastLanguageModel.from_pretrained(
    model_name=base_model_name,
    max_seq_length=4096,
    load_in_4bit=True
)

model = PeftModel.from_pretrained(base_model, adapter_path)


model.push_to_hub("TheCasvi/Qwen3-4B-CodeMedic-adapter", token=HUGGINGFACEHUB_API_TOKEN)


tokenizer = AutoTokenizer.from_pretrained(adapter_path, trust_remote_code=True)
tokenizer.push_to_hub("TheCasvi/Qwen3-4B-CodeMedic-adapter", token=HUGGINGFACEHUB_API_TOKEN)

/tmp/ipykernel_197032/2772276325.py:5: UserWarning: WARNING: Unsloth should be imported before transformers, peft to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
model_path:  ./results/final_checkpoint
==((====))==  Unsloth 2025.5.7: Fast Qwen3 patching. Transformers: 4.51.3.
   \\   /|    NVIDIA GeForce RTX 4070 SUPER. Num GPUs = 2. Max memory: 11.994 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.3.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


adapter_model.safetensors:   0%|          | 0.00/529M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/TheCasvi/Qwen3-4B-CodeMedic-adapter/commit/e945e314e33e9eef988b46bf95a312beedf90468', commit_message='Upload tokenizer', commit_description='', oid='e945e314e33e9eef988b46bf95a312beedf90468', pr_url=None, repo_url=RepoUrl('https://huggingface.co/TheCasvi/Qwen3-4B-CodeMedic-adapter', endpoint='https://huggingface.co', repo_type='model', repo_id='TheCasvi/Qwen3-4B-CodeMedic-adapter'), pr_revision=None, pr_num=None)

<h3>Adding the adapters to the base LLM for inference</h3>

In [ ]:
from unsloth import FastLanguageModel
from transformers import AutoTokenizer
from peft import PeftModel
import os
base_model_name="unsloth/Qwen3-4B-unsloth-bnb-4bit"
output_dir='./results'

base_model, _ = FastLanguageModel.from_pretrained(
    model_name = base_model_name, # MODEL USED FOR TRAINING
    max_seq_length = 512,
    load_in_4bit = True,
)

adapter_path = os.path.join(output_dir, "final_checkpoint")
tokenizer = AutoTokenizer.from_pretrained(adapter_path, trust_remote_code=True)
model = PeftModel.from_pretrained(base_model, adapter_path)


#merged_model = model.merge_and_unload()

messages = [
{"role" : "user", "content" : """
Solve the issue:
my_arr=[1,2,3,4,5,6,7,8,9]
i=0
while i>len(my_arr):
    print(my_arr[i])
    i=i-1"""}
]

text = tokenizer.apply_chat_template(
messages,
tokenize = False,
add_generation_prompt = True,
enable_thinking = False,
)

from transformers import TextStreamer
_ = model.generate(
**tokenizer(text, return_tensors = "pt").to("cuda"),
max_new_tokens = 256, # Increase for longer outputs!
temperature = 0.7, top_p = 0.8, top_k = 20, # For non thinking
streamer = TextStreamer(tokenizer, skip_prompt = True),
)

<h3>Doing inference with the model from hugging face using langchain</h3>

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
model_id="TheCasvi/Qwen3-4B-CodeMedic-adapter"
llm = HuggingFacePipeline.from_model_id(
    model_id=model_id,
    task="text-generation",
    pipeline_kwargs={
        "max_new_tokens": 1000,
        "do_sample": False,
        "repetition_penalty": 1.03,
    }
)
chat_model = ChatHuggingFace(llm=llm, model_id=model_id)
messages = [
    SystemMessage(content="You're a helpful code assistant"),
    HumanMessage(
        content="""Solve the issue:
my_arr=[1,2,3,4,5,6,7,8,9]
i=0
while i>len(my_arr):
    print(my_arr[i])
    i=i-1"""""
    ),
]

ai_msg = chat_model.invoke(messages)

print(ai_msg.content)